教学文件和生成内容保存在 /root/gpufree-data/PATHAK/tutorials/。选择 Python (pathak) 内核，按顺序执行。

PROJECT_ROOT 指向项目根目录，TUTORIALS_ROOT 指向教学目录。BioBERT 完整模型位于 models/sent_biobert_pubmed_base_cased/，Java JAR 和分句资源位于 resources/。

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/root/gpufree-data/PATHAK")
TUTORIALS_ROOT = PROJECT_ROOT / "tutorials"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import pathak_runtime
importlib.reload(pathak_runtime)
from pathak_runtime import configure
configure()
print("Project:", PROJECT_ROOT)
print("Tutorials:", TUTORIALS_ROOT)
print("Python:", sys.executable)


In [3]:
import subprocess
from importlib.metadata import version
from nltk.tokenize import sent_tokenize

for name in ["pyspark", "spark-nlp", "nlu", "numpy", "pandas"]:
    print(name, version(name))
subprocess.run([str(Path(os.environ["JAVA_HOME"]) / "bin/java"), "-version"], check=True)
assert len(sent_tokenize("The gene regulates roots. It affects flowering.")) == 2
print("NLTK check passed")


pyspark 3.2.3
spark-nlp 4.0.2
nlu 4.0.0
numpy 1.23.5
pandas 1.5.2
NLTK check passed


openjdk version "1.8.0_472"
OpenJDK Runtime Environment (Zulu 8.90.0.19-CA-linux64) (build 1.8.0_472-b08)
OpenJDK 64-Bit Server VM (Zulu 8.90.0.19-CA-linux64) (build 25.472-b08, mixed mode)


In [4]:
from pathak_runtime import start_spark, load_biobert, embedding_column
import numpy as np

spark = start_spark()
try:
    print("Spark:", spark.version, spark.sparkContext.master)
    pipe = load_biobert()
    texts = ["The gene regulates root development.", "The protein participates in flowering."]
    result = pipe.predict(texts, output_level="document")
    column = embedding_column(result)
    vectors = np.stack(result[column].to_numpy())
    assert vectors.shape == (2, 768), vectors.shape
    assert np.isfinite(vectors).all()
    assert np.linalg.norm(vectors, axis=1).min() > 0
    print("Embedding column:", column)
    print("Embedding shape:", vectors.shape)
    print("BioBERT check passed")
finally:
    spark.stop()


26/10/20 22:09:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark: 3.2.3 local[2]


2026-10-20 22:09:12.588724: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Embedding column: sentence_embedding_biobert
Embedding shape: (2, 768)
BioBERT check passed
